# DAEMON-Kaggle: Graph Neural Networks for Related Product Recommendations

**Self-contained notebook** — all source code inlined. No external `src/` imports.
Drop into Kaggle, enable GPU + Internet, and Run All.

Implements the DAEMON model from *Virinchi et al., ECML-PKDD 2022*:
- Directed asymmetric GNN for co-purchase and co-view product graphs
- Dual source/target embeddings with per-component ReLU message passing
- Three-component loss: co-purchase likelihood + asymmetry enforcement + co-view similarity
- FAISS-based recommendation serving with cold-start support

**WARNING: Enable Internet and GPU in Kaggle settings before running**
(Settings > Internet > ON, Accelerator > GPU T4 x2)

In [ ]:
# Cell 2: Package Installation
# WARNING: Enable Internet in Kaggle settings (Settings > Internet > ON)
# Dynamically detect PyTorch/CUDA version for correct DGL wheel

import torch
tv = torch.__version__.split("+")[0]
cv = "cu" + torch.version.cuda.replace(".", "")
dgl_url = f"https://data.dgl.ai/wheels/torch-{tv}/{cv}/repo.html"
print(f"PyTorch {tv}, CUDA {cv}")
print(f"DGL URL: {dgl_url}")

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "dgl", "-f", dgl_url, "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-gpu", "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "matplotlib", "tqdm", "scikit-learn", "-q"])

# Optional: for real product text features
# subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])

print("Dependencies installed")


In [ ]:
# Cell 3: All imports + GPU check + seeds
import sys, os, json, math, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize as sk_normalize

# --- DGL import (must come before any class using dgl.DGLBlock) ---
try:
    import dgl
    from dgl.dataloading import DataLoader as DGLDataLoader, MultiLayerNeighborSampler
    from dgl import function as fn
    print(f"DGL {dgl.__version__} imported")
except ImportError as e:
    raise ImportError(f"DGL not found: {e}. Check Cell 2 installation.")

# --- FAISS import ---
try:
    import faiss
    print(f"FAISS imported (GPU: {faiss.get_num_gpus()} GPUs)")
except ImportError:
    print("FAISS not available - will use brute-force fallback")

# --- GPU verification ---
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: Running on CPU - training will be very slow")
    print("Enable GPU accelerator on Kaggle (Accelerator > GPU T4 x2)")

# --- Reproducibility ---
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    dgl.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# Quick GPU memory report
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU Memory | allocated={allocated:.1f}GB reserved={reserved:.1f}GB")

print("All imports successful")


In [ ]:
# Cell 4: DAEMONConfig + DAEMONLayer + DAEMONModel + AsymmetricLoss + count_parameters
# Inlined from src/daemon_model.py
# NOTE: import dgl must happen BEFORE this cell (done in Cell 3)

# ============================================================================
# DAEMONConfig
# ============================================================================

@dataclass
class DAEMONConfig:
    """Single source of truth for all hyperparameters."""
    # Graph
    num_nodes: int = 0
    num_edges: int = 0
    num_relations: int = 2

    # Model architecture
    in_feats: int = 384
    hidden_dim: int = 128
    out_dim: int = 64
    num_layers: int = 3
    dropout: float = 0.1

    # Training
    epochs: int = 30
    batch_size: int = 1024
    num_neighbors: Tuple[int, ...] = (20, 10, 10)
    lr: float = 1e-4
    weight_decay: float = 1e-5
    grad_accum_steps: int = 1
    use_amp: bool = True
    patience: int = 5
    grad_clip_norm: float = 1.0

    # Loss
    num_neg: int = 5

    # Evaluation
    hitrate_k: Tuple[int, ...] = (5, 10, 20)
    val_every: int = 1

    # Paths
    data_dir: str = "/kaggle/input/daemon-data"
    output_dir: str = "/kaggle/working/"
    checkpoint_path: str = "/kaggle/working/daemon_best.pt"

    # Memory
    cleanup_every_n_epochs: int = 4

    def __post_init__(self):
        assert self.out_dim <= self.hidden_dim, \
            f"out_dim ({self.out_dim}) should be <= hidden_dim ({self.hidden_dim})"


# ============================================================================
# DAEMONLayer
# ============================================================================

class DAEMONLayer(nn.Module):
    """Single DAEMON layer with dual-embedding message passing.

    For each node u, the layer produces:
        h_s^l = ReLU(W^l @ AGG_cp_out(h_t^{l-1})) + ReLU(W^l @ AGG_cv_out(h_s^{l-1}))
        h_t^l = ReLU(W^l @ AGG_cp_in( h_s^{l-1})) + ReLU(W^l @ AGG_cv_in( h_t^{l-1}))

    ReLU is applied PER-COMPONENT before summing (matching Algorithm 1, line 4).
    Source embedding aggregates from OUT-neighbors (via dgl.reverse),
    target embedding aggregates from IN-neighbors (original sampled block).
    Both outputs are L2-normalised before return.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        block: dgl.DGLBlock,
        h_src: torch.Tensor,
        h_tgt: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        h_src = self.dropout(h_src)
        h_tgt = self.dropout(h_tgt)

        with block.local_scope():
            block.srcdata['h_src'] = h_src
            block.srcdata['h_tgt'] = h_tgt

            # Edge-type masks on the original (in-neighbour) block
            cp_eid = (block.edata['type'] == 0).nonzero(as_tuple=True)[0]
            cv_eid = (block.edata['type'] == 1).nonzero(as_tuple=True)[0]

            # ---- Source embedding update (OUT-neighbours) ----
            # Reverse the block so edges point from batch nodes *to* their
            # out-neighbours. We then aggregate dst→src (see _agg_rev).
            rev_block = dgl.reverse(block)
            rev_cp_eid = (rev_block.edata['type'] == 0).nonzero(as_tuple=True)[0]
            rev_cv_eid = (rev_block.edata['type'] == 1).nonzero(as_tuple=True)[0]

            h_s_cp = self._agg_rev(rev_block, rev_cp_eid, 'h_tgt')
            h_s_cv = self._agg_rev(rev_block, rev_cv_eid, 'h_src')

            # ---- Target embedding update (IN-neighbours) ----
            h_t_cp = self._aggregate(block, cp_eid, 'h_src')
            h_t_cv = self._aggregate(block, cv_eid, 'h_tgt')

            dst_ids = block.dstdata['_ID']

            # Apply shared weight + ReLU **per component** (Algorithm 1, line 4)
            h_s_new = F.relu(self.W(h_s_cp[dst_ids])) + F.relu(self.W(h_s_cv[dst_ids]))
            h_t_new = F.relu(self.W(h_t_cp[dst_ids])) + F.relu(self.W(h_t_cv[dst_ids]))

        # L2 normalise
        h_s_new = F.normalize(h_s_new, p=2, dim=-1)
        h_t_new = F.normalize(h_t_new, p=2, dim=-1)

        return h_s_new, h_t_new

    def _aggregate(
        self,
        block: dgl.DGLBlock,
        eid: torch.Tensor,
        src_feat_name: str,
    ) -> torch.Tensor:
        """Message passing over edges eid (src → dst)."""
        if eid.numel() == 0:
            return torch.zeros(
                block.num_dst_nodes(), self.W.out_features,
                device=block.device
            )
        subg = block.edge_subgraph(eid, preserve_nodes=True)
        subg.srcdata['x'] = subg.srcdata[src_feat_name]
        subg.update_all(
            fn.copy_u('x', 'm'),
            fn.mean('m', 'agg'),
        )
        return subg.dstdata['agg']

    def _agg_rev(
        self,
        block: dgl.DGLBlock,
        eid: torch.Tensor,
        feat_name: str,
    ) -> torch.Tensor:
        """Message passing aggregating from dst to src.
        Used for source-embedding updates where out-neighbour features
        reside in block.dstdata after a dgl.reverse."""
        if eid.numel() == 0:
            return torch.zeros(
                block.num_src_nodes(), self.W.out_features,
                device=block.device
            )
        subg = block.edge_subgraph(eid, preserve_nodes=True)
        feat = subg.dstdata[feat_name]
        subg_rev = dgl.reverse(subg)
        subg_rev.srcdata['x'] = feat
        subg_rev.update_all(
            fn.copy_u('x', 'm'),
            fn.mean('m', 'agg'),
        )
        return subg_rev.dstdata['agg']


# ============================================================================
# DAEMONModel
# ============================================================================

class DAEMONModel(nn.Module):
    """Full DAEMON model: stacked DAEMONLayer → projection → L2 norm."""
    def __init__(self, cfg: DAEMONConfig):
        super().__init__()
        self.cfg = cfg

        # Input projection (if features differ from hidden dim)
        self.input_proj: Optional[nn.Linear] = None
        if cfg.in_feats != cfg.hidden_dim:
            self.input_proj = nn.Linear(cfg.in_feats, cfg.hidden_dim)

        # GNN layers
        self.layers = nn.ModuleList()
        for i in range(cfg.num_layers):
            in_dim = cfg.hidden_dim
            out_dim = cfg.hidden_dim
            self.layers.append(
                DAEMONLayer(in_dim, out_dim, dropout=cfg.dropout)
            )

        # Output projection to final embedding dimension
        self.out_proj = nn.Linear(cfg.hidden_dim, cfg.out_dim)

    def forward(
        self,
        blocks: List[dgl.DGLBlock],
        h: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Input projection
        if self.input_proj is not None:
            h = self.input_proj(h)

        h_src, h_tgt = h, h  # initialise both from same input

        for layer, block in zip(self.layers, blocks):
            h_src, h_tgt = layer(block, h_src, h_tgt)

        # Output projection + L2 norm
        h_src = F.normalize(self.out_proj(h_src), p=2, dim=-1)
        h_tgt = F.normalize(self.out_proj(h_tgt), p=2, dim=-1)

        return h_src, h_tgt


# ============================================================================
# AsymmetricLoss
# ============================================================================

class AsymmetricLoss(nn.Module):
    """Three-component loss for directed graph recommendation (Eq. 2 in paper).
    Components:
        1. Co-purchase likelihood (positive sampling + negative sampling)
        2. Asymmetry enforcement (one-way edge penalty)
        3. Co-view similarity (source + target alignment)"""
    def __init__(self, cfg: DAEMONConfig):
        super().__init__()
        self.num_neg = cfg.num_neg

    def forward(
        self,
        src_emb: torch.Tensor,
        tgt_emb: torch.Tensor,
        cp_u: torch.Tensor,
        cp_v: torch.Tensor,
        ow_u: torch.Tensor,
        ow_v: torch.Tensor,
        cv_u: torch.Tensor,
        cv_v: torch.Tensor,
        num_nodes: int,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        device = src_emb.device
        components = {}

        # Component 1: Co-purchase likelihood
        loss_cp = self._co_purchase_loss(src_emb, tgt_emb, cp_u, cp_v, num_nodes)
        components['cp'] = loss_cp.detach()

        # Component 2: Asymmetry enforcement
        loss_ow = self._asymmetry_loss(src_emb, tgt_emb, ow_u, ow_v)
        components['ow'] = loss_ow.detach()

        # Component 3: Co-view similarity
        loss_cv = self._co_view_loss(src_emb, tgt_emb, cv_u, cv_v)
        components['cv'] = loss_cv.detach()

        total = loss_cp + loss_ow + loss_cv
        return total, components

    def _co_purchase_loss(
        self,
        src_emb: torch.Tensor,
        tgt_emb: torch.Tensor,
        u: torch.Tensor,
        v: torch.Tensor,
        num_nodes: int,
    ) -> torch.Tensor:
        if u.numel() == 0:
            return torch.tensor(0.0, device=src_emb.device)

        pos = (src_emb[u] * tgt_emb[v]).sum(dim=1)
        loss = -F.logsigmoid(pos).mean()

        # Negative sampling
        neg_z = torch.randint(0, num_nodes, (u.numel(), self.num_neg), device=src_emb.device)
        neg = (src_emb[u].unsqueeze(1) * tgt_emb[neg_z]).sum(dim=2)
        loss = loss + (-F.logsigmoid(-neg)).mean()

        return loss

    def _asymmetry_loss(
        self,
        src_emb: torch.Tensor,
        tgt_emb: torch.Tensor,
        u: torch.Tensor,
        v: torch.Tensor,
    ) -> torch.Tensor:
        if u.numel() == 0:
            return torch.tensor(0.0, device=src_emb.device)

        forward = (src_emb[u] * tgt_emb[v]).sum(dim=1)
        reverse = (src_emb[v] * tgt_emb[u]).sum(dim=1)

        loss = -F.logsigmoid(forward).mean() - F.logsigmoid(-reverse).mean()
        return loss

    def _co_view_loss(
        self,
        src_emb: torch.Tensor,
        tgt_emb: torch.Tensor,
        u: torch.Tensor,
        v: torch.Tensor,
    ) -> torch.Tensor:
        if u.numel() == 0:
            return torch.tensor(0.0, device=src_emb.device)

        src_sim = (src_emb[u] * src_emb[v]).sum(dim=1)
        tgt_sim = (tgt_emb[u] * tgt_emb[v]).sum(dim=1)

        loss = -F.logsigmoid(src_sim).mean() - F.logsigmoid(tgt_sim).mean()
        return loss


# ============================================================================
# Utility: count parameters
# ============================================================================

def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
# Cell 5: Data pipeline functions
# Inlined from src/data_pipeline.py
# Key functions: generate_synthetic_graph, build_product_graph, print_graph_stats,
#                split_edges_by_type, find_one_way_edges, validate_graph, NegativeSampler,
#                estimate_asymmetry
# NOTE: Text encoding functions (encode_product_texts, encode_lightweight, etc.)
#       are skipped — not needed for synthetic data test.
#       Uncomment if using real product text data.

def _power_law_degrees(n: int, avg_degree: float, rng: np.random.Generator) -> np.ndarray:
    """Sample a power-law degree sequence with target mean."""
    a = 2.2
    raw = rng.zipf(a, size=n).astype(np.float64)
    raw = np.clip(raw, 1, n - 1)
    degrees = (raw / raw.mean() * avg_degree).round().astype(np.int64)
    degrees = np.clip(degrees, 1, n - 1)
    if degrees.sum() % 2 != 0:
        degrees[0] += 1
        degrees = np.clip(degrees, 1, n - 1)
    return degrees


def generate_synthetic_graph(
    num_products: int = 10000,
    feature_dim: int = 384,
    avg_cp_degree: int = 5,
    avg_cv_degree: int = 8,
    asymmetry_ratio: float = 0.75,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Generate a synthetic product graph for testing DAEMON."""
    rng = np.random.default_rng(seed)

    # 2a. Product features — cluster-like to simulate categories
    num_categories = max(1, num_products // 500)
    cluster_centers = rng.standard_normal((num_categories, feature_dim)).astype(np.float32)
    categories = rng.integers(0, num_categories, size=num_products)
    features = (
        cluster_centers[categories]
        + 0.3 * rng.standard_normal((num_products, feature_dim)).astype(np.float32)
    )
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features = features / np.clip(norms, 1e-8, None)

    # 2b. Per-category product lists
    cat_to_products = [np.where(categories == c)[0] for c in range(num_categories)]
    cp_deg = _power_law_degrees(num_products, avg_cp_degree, rng)
    cv_deg = _power_law_degrees(num_products, avg_cv_degree, rng)

    # 2c. Co-purchase edges (within-category, asymmetric)
    cp_src, cp_dst = [], []
    for u in range(num_products):
        cat = categories[u]
        neighbours = cat_to_products[cat]
        candidates = neighbours[neighbours != u]
        if len(candidates) == 0:
            continue
        deg = int(cp_deg[u])
        if deg == 0:
            continue
        k = min(deg, len(candidates))
        targets = rng.choice(candidates, size=k, replace=False)
        for v in targets.tolist():
            cp_src.append(u)
            cp_dst.append(v)
            if rng.random() > asymmetry_ratio:
                cp_src.append(v)
                cp_dst.append(u)

    # 2d. Co-view edges (bidirectional + cross-category)
    cv_src, cv_dst = [], []
    for u in range(num_products):
        cat = categories[u]
        deg = int(cv_deg[u])
        if deg == 0:
            continue
        n_within = max(0, deg * 3 // 5)
        neighbours = cat_to_products[cat]
        candidates_within = neighbours[neighbours != u]
        if len(candidates_within) > 0:
            k = min(n_within, len(candidates_within))
            within_targets = rng.choice(candidates_within, size=k, replace=False)
            for v in within_targets.tolist():
                cv_src.append(u); cv_dst.append(v)
                cv_src.append(v); cv_dst.append(u)
        n_cross = deg - n_within
        other_cats = [c for c in range(num_categories) if c != cat]
        if other_cats and n_cross > 0:
            other_cat = rng.choice(other_cats)
            other_neighbours = cat_to_products[other_cat]
            if len(other_neighbours) > 0:
                k = min(n_cross, len(other_neighbours))
                cross_targets = rng.choice(other_neighbours, size=k, replace=False)
                for v in cross_targets.tolist():
                    cv_src.append(u); cv_dst.append(v)
                    cv_src.append(v); cv_dst.append(u)

    # 2e. Convert to numpy arrays
    edge_index_cp = np.array([cp_src, cp_dst], dtype=np.int64) if cp_src else np.zeros((2, 0), dtype=np.int64)
    edge_index_cv = np.array([cv_src, cv_dst], dtype=np.int64) if cv_src else np.zeros((2, 0), dtype=np.int64)

    asymmetry = estimate_asymmetry(edge_index_cp)
    print(f"Generated: {num_products} nodes, {edge_index_cp.shape[1]:,} CP edges, {edge_index_cv.shape[1]:,} CV edges")
    print(f"  Co-purchase asymmetry: {asymmetry:.1f}% one-way edges")
    print(f"  Features shape: {features.shape}")

    return edge_index_cp, edge_index_cv, features, categories


def estimate_asymmetry(edges: np.ndarray) -> float:
    """Estimate the percentage of one-way (directed) edges."""
    if edges.shape[1] == 0:
        return 0.0
    edge_set = set(zip(edges[0].tolist(), edges[1].tolist()))
    one_way = sum(1 for u, v in zip(edges[0].tolist(), edges[1].tolist()) if (v, u) not in edge_set)
    return 100.0 * one_way / edges.shape[1]


def build_product_graph(
    product_df: "pd.DataFrame",
    cp_edges: np.ndarray,
    cv_edges: np.ndarray,
    features: Optional[np.ndarray] = None,
    feature_dim: int = 384,
) -> dgl.DGLGraph:
    """Build a DGL directed graph from co-purchase and co-view edges.
    Returns the DGL graph (not a tuple)."""
    N = len(product_df)

    # Obtain node features
    if features is not None:
        features_np = np.asarray(features, dtype=np.float32)
    else:
        # Fallback: random features
        rng = np.random.default_rng(42)
        features_np = rng.standard_normal((N, feature_dim), dtype=np.float32)

    # Combine edges with type annotations
    all_src = np.concatenate([cp_edges[0], cv_edges[0]])
    all_dst = np.concatenate([cp_edges[1], cv_edges[1]])
    edge_types = np.concatenate([
        np.zeros(cp_edges.shape[1], dtype=np.int64),
        np.ones(cv_edges.shape[1], dtype=np.int64),
    ])

    # Build DGL graph
    g = dgl.graph((all_src, all_dst), num_nodes=N)
    g.edata["type"] = torch.as_tensor(edge_types, dtype=torch.long)
    g.ndata["feat"] = torch.as_tensor(features_np, dtype=torch.float32)

    directed_pct = estimate_directed_pct(g)
    print(f"Graph built: {N:,} nodes, {g.num_edges():,} edges")
    print(f"  Co-purchase: {cp_edges.shape[1]:,} edges")
    print(f"  Co-view:     {cv_edges.shape[1]:,} edges")
    print(f"  Directed:    {directed_pct:.1f}%")

    return g


def estimate_directed_pct(g: dgl.DGLGraph) -> float:
    """Percentage of directed (one-way) edges in a DGL graph."""
    src, dst = g.edges()
    edge_set = set(zip(src.tolist(), dst.tolist()))
    one_way = sum(1 for s, d in zip(src.tolist(), dst.tolist()) if (d, s) not in edge_set)
    return 100.0 * one_way / g.num_edges()


def print_graph_stats(g: dgl.DGLGraph) -> None:
    """Print detailed graph statistics to stdout."""
    in_degrees = g.in_degrees().float()
    out_degrees = g.out_degrees().float()
    cp_mask = g.edata["type"] == 0
    cv_mask = g.edata["type"] == 1
    src, dst = g.edges()
    edge_set = set(zip(src.tolist(), dst.tolist()))
    bidirectional = sum(1 for s, d in zip(src.tolist(), dst.tolist()) if (d, s) in edge_set) // 2
    one_way = g.num_edges() - 2 * bidirectional

    print("=" * 52)
    print("  GRAPH STATISTICS")
    print("=" * 52)
    print(f"  Nodes:           {g.num_nodes():>12,}")
    print(f"  Total edges:     {g.num_edges():>12,}")
    print(f"    Co-purchase:   {int(cp_mask.sum()):>12,}")
    print(f"    Co-view:       {int(cv_mask.sum()):>12,}")
    print(f"  Avg degree:      {float(in_degrees.mean()):>12.1f}")
    print(f"  Median degree:   {float(in_degrees.median()):>12.1f}")
    print(f"  Max degree:      {int(in_degrees.max()):>12,}")
    print(f"  Min degree:      {int(in_degrees.min()):>12,}")
    print(f"  Isolated nodes:  {int((in_degrees == 0).sum()):>12,}")
    print(f"  Bidirectional:   {bidirectional:>12,}")
    print(f"  One-way:         {one_way:>12,}")
    print(f"  Directed:        {100.0 * one_way / g.num_edges():>11.1f}%")
    print(f"  Feature dim:     {g.ndata['feat'].shape[1]:>12}")
    print(f"  Feature NaN:     {bool(torch.isnan(g.ndata['feat']).any()):>12}")
    print(f"  Feature Inf:     {bool(torch.isinf(g.ndata['feat']).any()):>12}")
    print("=" * 52)


def split_edges_by_type(
    g: dgl.DGLGraph,
    train_ratio: float = 0.75,
    val_ratio: float = 0.05,
) -> Dict[str, torch.Tensor]:
    """Split edges into train / val / test sets, stratified by edge type."""
    cp_mask = g.edata["type"] == 0
    cv_mask = g.edata["type"] == 1

    def _split_mask(mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        eids = mask.nonzero(as_tuple=True)[0]
        perm = eids[torch.randperm(len(eids))]
        n_train = int(len(perm) * train_ratio)
        n_val = int(len(perm) * val_ratio)
        return perm[:n_train], perm[n_train:n_train + n_val], perm[n_train + n_val:]

    train_cp, val_cp, test_cp = _split_mask(cp_mask)
    train_cv, val_cv, test_cv = _split_mask(cv_mask)

    return {
        "train_cp": train_cp, "train_cv": train_cv,
        "val_cp": val_cp, "val_cv": val_cv,
        "test_cp": test_cp, "test_cv": test_cv,
    }


def find_one_way_edges(g: dgl.DGLGraph) -> Tuple[torch.Tensor, torch.Tensor]:
    """Find one-way co-purchase edges for asymmetry loss computation."""
    src, dst = g.edges()
    cp_mask = g.edata["type"] == 0
    cp_src = src[cp_mask]
    cp_dst = dst[cp_mask]
    edge_set = set(zip(cp_src.tolist(), cp_dst.tolist()))
    one_way_mask = torch.tensor(
        [(dst_i.item(), src_i.item()) not in edge_set for src_i, dst_i in zip(cp_src, cp_dst)],
        device=src.device,
    )
    return cp_src[one_way_mask], cp_dst[one_way_mask]


def validate_graph(g: dgl.DGLGraph) -> bool:
    """Comprehensive graph validation."""
    N = g.num_nodes()
    E = g.num_edges()
    assert N > 0, "Graph has 0 nodes"
    assert "feat" in g.ndata, "No node features"
    feat = g.ndata["feat"]
    assert feat.shape[0] == N, f"Feature rows ({feat.shape[0]}) != nodes ({N})"
    assert not torch.isnan(feat).any(), "NaN in node features"
    assert not torch.isinf(feat).any(), "Inf in node features"
    assert E > 0, "Graph has 0 edges"
    assert "type" in g.edata, "No edge types"
    unique_types = set(g.edata["type"].unique().tolist())
    assert unique_types.issubset({0, 1}), f"Unexpected edge types: {unique_types}"
    src, dst = g.edges()
    assert src.min() >= 0 and src.max() < N, "Source out of bounds"
    assert dst.min() >= 0 and dst.max() < N, "Destination out of bounds"
    edge_set = set(zip(src.tolist(), dst.tolist()))
    one_way = sum(1 for s, d in zip(src.tolist(), dst.tolist()) if (d, s) not in edge_set)
    one_way_pct = 100.0 * one_way / E
    assert one_way_pct > 10, f"Only {one_way_pct:.1f}% edges are directed"
    print(f"Graph validated: {N:,} nodes, {E:,} edges, {one_way_pct:.1f}% directed")
    return True


class NegativeSampler:
    """Uniform random negative sampler for link prediction training."""
    def __init__(self, num_nodes: int, num_neg: int = 5, device: Union[str, torch.device] = "cpu"):
        self.num_nodes = num_nodes
        self.num_neg = num_neg
        self.device = torch.device(device)

    def sample(self, num_positives: int) -> torch.Tensor:
        return torch.randint(0, self.num_nodes, (num_positives, self.num_neg), device=self.device)


print("Data pipeline functions defined")


In [ ]:
# Cell 6: Training functions
# Inlined from src/training.py — simplified for notebook usage

# ---------------------------------------------------------------------------
# Memory utilities
# ---------------------------------------------------------------------------

def memory_summary() -> Dict[str, float]:
    """Return GPU memory statistics and print summary."""
    if not torch.cuda.is_available():
        stats = {"allocated_gb": 0.0, "reserved_gb": 0.0, "peak_gb": 0.0, "free_gb": 0.0}
    else:
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        stats = {
            "allocated_gb": round(allocated, 2),
            "reserved_gb": round(reserved, 2),
            "peak_gb": round(peak, 2),
            "free_gb": round(total - allocated, 2),
        }
    print(f"GPU Memory | allocated={stats['allocated_gb']:.1f}GB reserved={stats['reserved_gb']:.1f}GB peak={stats['peak_gb']:.1f}GB free={stats['free_gb']:.1f}GB")
    return stats


# ---------------------------------------------------------------------------
# Batch data builder
# ---------------------------------------------------------------------------

def _build_default_batch_data(block) -> Dict[str, torch.Tensor]:
    """Build a minimal batch_data dict from a DGL block."""
    src, dst = block.edges()
    device = src.device
    if "type" in block.edata:
        cp_mask = block.edata["type"] == 0
        cv_mask = block.edata["type"] == 1
        cp_src = src[cp_mask]
        cp_dst = dst[cp_mask]
        cv_src = src[cv_mask]
        cv_dst = dst[cv_mask]
    else:
        cp_src, cp_dst = src, dst
        cv_src = cv_dst = torch.empty(0, dtype=torch.long, device=device)
    num_nodes = block.num_dst_nodes()
    return {
        "cp_u": cp_src, "cp_v": cp_dst,
        "ow_u": torch.empty(0, dtype=torch.long, device=device),
        "ow_v": torch.empty(0, dtype=torch.long, device=device),
        "cv_u": cv_src, "cv_v": cv_dst,
        "num_nodes": num_nodes,
    }


# ---------------------------------------------------------------------------
# Epoch-level training
# ---------------------------------------------------------------------------

def train_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    cfg: Any,
    epoch_idx: int = 0,
) -> float:
    """Run one training epoch with AMP, gradient accumulation, and clipping."""
    model.train()
    total_loss: float = 0.0
    num_batches: int = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f"Epoch {epoch_idx:3d}", leave=False)
    train_device = next(model.parameters()).device
    accum_steps = getattr(cfg, "grad_accum_steps", 1)

    for step, batch in enumerate(pbar):
        input_nodes, output_nodes, blocks = batch
        blocks = [b.to(train_device, non_blocking=True) for b in blocks]
        batch_inputs = blocks[0].srcdata["feat"]

        with autocast(enabled=getattr(cfg, "use_amp", True)):
            src_emb, tgt_emb = model(blocks, batch_inputs)

            batch_data = _build_default_batch_data(blocks[-1])
            # Unpack dict to match AsymmetricLoss.forward signature
            loss, _ = criterion(
                src_emb, tgt_emb,
                cp_u=batch_data["cp_u"], cp_v=batch_data["cp_v"],
                ow_u=batch_data["ow_u"], ow_v=batch_data["ow_v"],
                cv_u=batch_data["cv_u"], cv_v=batch_data["cv_v"],
                num_nodes=batch_data["num_nodes"],
            )
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            clip_norm = getattr(cfg, "grad_clip_norm", 1.0)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        batch_loss = loss.item() * accum_steps
        total_loss += batch_loss
        num_batches += 1
        pbar.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            vram=f"{torch.cuda.memory_allocated() / 1e9:.1f}G",
        )

    return total_loss / max(num_batches, 1)


# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

@torch.no_grad()
def validate_model(
    model: nn.Module,
    g: dgl.DGLGraph,
    val_cp_eids: torch.Tensor,
    device: str = "cuda",
) -> Dict[str, float]:
    """Quick validation: generate embeddings and compute AUC + HR@10 on val edges."""
    model.eval()
    src_emb, tgt_emb = generate_all_embeddings(model, g, batch_size=4096, device=device)
    src_emb = src_emb.to(device)
    tgt_emb = tgt_emb.to(device)

    num_val = len(val_cp_eids)
    if num_val == 0:
        return {"auc": 0.0, "hitrate_10": 0.0}

    val_src, val_dst = g.find_edges(val_cp_eids)
    val_src = val_src.to(device)
    val_dst = val_dst.to(device)

    num_nodes = g.num_nodes()
    neg_src = torch.randint(0, num_nodes, (num_val,), device=device)
    neg_dst = torch.randint(0, num_nodes, (num_val,), device=device)

    pos_scores = (src_emb[val_src] * tgt_emb[val_dst]).sum(dim=1)
    neg_scores = (src_emb[neg_src] * tgt_emb[neg_dst]).sum(dim=1)

    scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
    labels = np.concatenate([np.ones(num_val), np.zeros(num_val)])
    auc = float(roc_auc_score(labels, scores))

    # HitRate@10
    hr10 = compute_hit_rate_at_k(src_emb, tgt_emb, val_src, val_dst, k=10)

    del src_emb, tgt_emb
    torch.cuda.empty_cache()
    return {"auc": auc, "hitrate_10": hr10}


# ---------------------------------------------------------------------------
# Checkpointing
# ---------------------------------------------------------------------------

def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    scheduler: Any,
    epoch: int,
    metrics: Dict[str, float],
    path: str,
    is_best: bool = False,
) -> str:
    """Save training checkpoint to disk."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "metrics": metrics,
    }
    torch.save(checkpoint, path)
    if is_best:
        best_path = path + ".best"
        torch.save(checkpoint, best_path)
        print(f"Best model saved -> {best_path}")
    return path


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler: Optional[GradScaler] = None,
    scheduler: Any = None,
    map_location: Optional[str] = None,
) -> Tuple[int, Dict[str, float]]:
    """Load a training checkpoint and restore all states."""
    if map_location is None:
        map_location = "cuda" if torch.cuda.is_available() else "cpu"
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scaler is not None and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        sd = checkpoint["scheduler_state_dict"]
        if sd is not None:
            scheduler.load_state_dict(sd)
    start_epoch = checkpoint.get("epoch", 0)
    metrics = checkpoint.get("metrics", {})
    print(f"Checkpoint loaded <- {path} (resume epoch {start_epoch}, best AUC={metrics.get('auc', 0):.4f})")
    return start_epoch + 1, metrics


# ---------------------------------------------------------------------------
# Setup training
# ---------------------------------------------------------------------------

def setup_training(cfg: Any) -> Dict[str, Any]:
    """Initialize all training components from config."""
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DAEMONModel(cfg).to(device)
    n_params = count_parameters(model)
    print(f"Model: DAEMONModel | Params: {n_params:,}")

    lr = getattr(cfg, "lr", 1e-4)
    weight_decay = getattr(cfg, "weight_decay", 1e-5)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay)

    criterion = AsymmetricLoss(cfg).to(device)

    # CosineAnnealingWarmRestarts scheduler
    T_0 = getattr(cfg, "scheduler_t0", 10)
    T_mult = getattr(cfg, "scheduler_tmult", 2)
    eta_min = getattr(cfg, "scheduler_eta_min", 1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T_0, T_mult=T_mult, eta_min=eta_min,
    )

    use_amp = getattr(cfg, "use_amp", True)
    scaler = GradScaler(enabled=use_amp)

    print(f"Optimizer: Adam (lr={lr:.2e}, wd={weight_decay:.1e})")
    print(f"Scheduler: CosineAnnealingWarmRestarts | AMP: {'enabled' if use_amp else 'disabled'} | Device: {device}")

    return {"model": model, "optimizer": optimizer, "criterion": criterion,
            "scheduler": scheduler, "scaler": scaler, "start_epoch": 0}


# ---------------------------------------------------------------------------
# Full training orchestration
# ---------------------------------------------------------------------------

def train_model(
    model: nn.Module,
    train_loader,
    g: dgl.DGLGraph,
    val_cp_eids: torch.Tensor,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    scaler: GradScaler,
    cfg: Any,
    resume_epoch: int = 0,
) -> Dict[str, List[float]]:
    """Run the full training loop with early stopping and best-model tracking.
    Returns history dict with train_loss, val_auc, val_hr10 lists."""
    num_epochs = getattr(cfg, "epochs", 30)
    patience = getattr(cfg, "patience", 5)
    val_every = getattr(cfg, "val_every", 1)
    checkpoint_path = getattr(cfg, "checkpoint_path", "/tmp/daemon_checkpoint.pt")
    cleanup_every = getattr(cfg, "cleanup_every_n_epochs", 5)
    device = next(model.parameters()).device

    best_val_auc = 0.0
    best_epoch = -1
    patience_counter = 0

    history: Dict[str, List[float]] = {"train_loss": [], "val_auc": [], "val_hr10": [], "lr": []}

    print(f"{'=' * 60}")
    print(f"  DAEMON Training — {num_epochs} max epochs, patience={patience}")
    print(f"{'=' * 60}")
    memory_summary()

    for epoch in range(resume_epoch, num_epochs):
        epoch_start = time.time()

        # Training
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, cfg, epoch_idx=epoch)
        history["train_loss"].append(train_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(current_lr)

        # Validation
        val_results = None
        if (epoch + 1) % val_every == 0 and len(val_cp_eids) > 0:
            val_results = validate_model(model, g, val_cp_eids, device=str(device))
            val_auc = val_results.get("auc", 0.0)
            val_hr10 = val_results.get("hitrate_10", 0.0)
            history["val_auc"].append(val_auc)
            history["val_hr10"].append(val_hr10)

            scheduler.step()

            is_best = val_auc > best_val_auc
            if is_best:
                best_val_auc = val_auc
                best_epoch = epoch
                patience_counter = 0
            else:
                patience_counter += 1

            save_checkpoint(model, optimizer, scaler, scheduler, epoch, val_results, checkpoint_path, is_best=is_best)

        # Epoch summary
        epoch_time = time.time() - epoch_start
        mem = memory_summary()
        parts = [f"Epoch {epoch:3d}", f"Loss {train_loss:.4f}"]
        if val_results:
            parts.append(f"AUC {val_results.get('auc', 0):.4f}")
            parts.append(f"HR@10 {val_results.get('hitrate_10', 0):.4f}")
        parts.append(f"LR {current_lr:.2e}")
        parts.append(f"Time {epoch_time:.1f}s")
        parts.append(f"Best {best_val_auc:.4f}")
        print("  | ".join(parts))

        # Memory cleanup
        if (epoch + 1) % cleanup_every == 0:
            gc.collect()
            torch.cuda.empty_cache()

        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch} (best AUC={best_val_auc:.4f} at epoch {best_epoch})")
            break

    print(f"Training Complete — {len(history['train_loss'])} epochs")
    print(f"Best Val AUC: {best_val_auc:.4f} (epoch {best_epoch})")
    return history


print("Training functions defined")


In [ ]:
# Cell 7: Evaluation + FAISS functions
# Inlined from src/evaluation.py

# ---------------------------------------------------------------------------
# Helper: ensure 2-column edge format
# ---------------------------------------------------------------------------

def _ensure_2col(edges: torch.Tensor) -> torch.Tensor:
    """Ensure edge tensor has shape [num_edges, 2]."""
    if edges.dim() == 2 and edges.size(0) == 2 and edges.size(1) != 2:
        edges = edges.T
    return edges


# ---------------------------------------------------------------------------
# Embedding generation
# ---------------------------------------------------------------------------

@torch.no_grad()
def generate_all_embeddings(
    model: torch.nn.Module,
    g: dgl.DGLGraph,
    batch_size: int = 4096,
    device: str = "cuda",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Run full-graph inference to generate source and target embeddings for all nodes."""
    model = model.to(device)
    model.eval()

    num_layers = len(model.layers) if hasattr(model, "layers") else 3
    sampler = MultiLayerNeighborSampler([-1] * num_layers)

    num_nodes = g.num_nodes()
    all_nodes = torch.arange(num_nodes)

    loader = DGLDataLoader(
        g, all_nodes, sampler,
        batch_size=batch_size, shuffle=False, drop_last=False, device=device,
    )

    src_list: List[torch.Tensor] = []
    tgt_list: List[torch.Tensor] = []

    for input_nodes, output_nodes, blocks in loader:
        h = blocks[0].srcdata["feat"]
        out = model(blocks, h)
        if isinstance(out, (tuple, list)) and len(out) == 2:
            src_list.append(out[0].cpu())
            tgt_list.append(out[1].cpu())
        else:
            src_list.append(out.cpu())

    if tgt_list:
        return torch.cat(src_list, dim=0), torch.cat(tgt_list, dim=0)
    return torch.cat(src_list, dim=0), torch.cat(src_list, dim=0)


# ---------------------------------------------------------------------------
# EQ1 — Node recommendation metrics
# ---------------------------------------------------------------------------

@torch.no_grad()
def compute_hit_rate_at_k(
    src_emb: torch.Tensor,
    tgt_emb: torch.Tensor,
    query_ids: torch.Tensor,
    true_candidate_ids: torch.Tensor,
    k: int = 10,
) -> float:
    """HitRate@k for node recommendation."""
    device = src_emb.device
    query_ids = query_ids.to(device)
    true_candidate_ids = true_candidate_ids.to(device)
    q_src = src_emb[query_ids]
    scores = q_src @ tgt_emb.T
    _, topk_indices = scores.topk(k=k, dim=1, largest=True, sorted=True)
    hits = (topk_indices == true_candidate_ids.unsqueeze(1)).any(dim=1).float()
    return float(hits.mean().item())


@torch.no_grad()
def compute_mrr_at_k(
    src_emb: torch.Tensor,
    tgt_emb: torch.Tensor,
    query_ids: torch.Tensor,
    true_candidate_ids: torch.Tensor,
    k: int = 10,
) -> float:
    """MRR@k (Mean Reciprocal Rank) for node recommendation."""
    device = src_emb.device
    query_ids = query_ids.to(device)
    true_candidate_ids = true_candidate_ids.to(device)
    q_src = src_emb[query_ids]
    scores = q_src @ tgt_emb.T
    effective_k = min(k, scores.size(1))
    _, topk_indices = scores.topk(k=effective_k, dim=1, largest=True, sorted=True)
    match = topk_indices == true_candidate_ids.unsqueeze(1)
    first_match_pos = match.int().argmax(dim=1)
    found = match.any(dim=1)
    ranks = torch.where(found, first_match_pos + 1, torch.zeros_like(first_match_pos))
    reciprocal_ranks = torch.where(found, 1.0 / ranks.float(), torch.zeros_like(ranks.float()))
    return float(reciprocal_ranks.mean().item())


# ---------------------------------------------------------------------------
# EQ2 — Link prediction AUC
# ---------------------------------------------------------------------------

@torch.no_grad()
def compute_link_prediction_auc(
    src_emb: torch.Tensor,
    tgt_emb: torch.Tensor,
    pos_edges: torch.Tensor,
    neg_edges: torch.Tensor,
) -> float:
    """ROC-AUC for existential link prediction."""
    pos_edges = _ensure_2col(pos_edges)
    neg_edges = _ensure_2col(neg_edges)
    pos_scores = (src_emb[pos_edges[:, 0]] * tgt_emb[pos_edges[:, 1]]).sum(dim=1)
    neg_scores = (src_emb[neg_edges[:, 0]] * tgt_emb[neg_edges[:, 1]]).sum(dim=1)
    scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
    labels = np.concatenate([np.ones(pos_scores.shape[0]), np.zeros(neg_scores.shape[0])])
    return float(roc_auc_score(labels, scores))


# ---------------------------------------------------------------------------
# EQ3 — Direction prediction AUC
# ---------------------------------------------------------------------------

@torch.no_grad()
def compute_direction_prediction_auc(
    src_emb: torch.Tensor,
    tgt_emb: torch.Tensor,
    one_way_edges: torch.Tensor,
) -> float:
    """ROC-AUC for direction prediction."""
    one_way_edges = _ensure_2col(one_way_edges)
    u = one_way_edges[:, 0]
    v = one_way_edges[:, 1]
    forward_scores = (src_emb[u] * tgt_emb[v]).sum(dim=1)
    reverse_scores = (src_emb[v] * tgt_emb[u]).sum(dim=1)
    scores = torch.cat([forward_scores, reverse_scores]).cpu().numpy()
    labels = np.concatenate([np.ones(forward_scores.shape[0]), np.zeros(reverse_scores.shape[0])])
    return float(roc_auc_score(labels, scores))


# ---------------------------------------------------------------------------
# FAISS index construction and search
# ---------------------------------------------------------------------------

def build_faiss_index(
    embeds: torch.Tensor,
    use_gpu: bool = True,
    nlist: Optional[int] = None,
    nprobe: int = 32,
):
    """Build a GPU FAISS IndexIVFFlat for cosine-similarity search."""
    embeds_np = embeds.cpu().numpy().astype(np.float32)
    faiss.normalize_L2(embeds_np)
    dim = embeds_np.shape[1]
    num_vectors = embeds_np.shape[0]
    if nlist is None:
        nlist = min(4096, int(np.sqrt(num_vectors)))
    nlist = max(nlist, 1)
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.nprobe = nprobe
    index.train(embeds_np)
    index.add(embeds_np)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        co = faiss.GpuClonerOptions()
        co.useFloat16 = True
        index = faiss.index_cpu_to_gpu(res, 0, index, co)
    return index


def recommend_related(
    embeds: torch.Tensor,
    index,
    product_idx: int,
    k: int = 10,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return the top-k most related products for a given query product."""
    query_vec = embeds[product_idx].cpu().numpy().astype(np.float32).reshape(1, -1)
    faiss.normalize_L2(query_vec)
    distances, indices = index.search(query_vec, k + 1)
    mask = indices[0] != product_idx
    result_indices = indices[0][mask][:k]
    result_distances = distances[0][mask][:k]
    if len(result_indices) < k:
        extra = k - len(result_indices)
        alt_mask = ~mask
        result_indices = np.concatenate([result_indices, indices[0][alt_mask][:extra]])
        result_distances = np.concatenate([result_distances, distances[0][alt_mask][:extra]])
    return result_indices, result_distances


# ---------------------------------------------------------------------------
# Cold-start recommendation (k-NN + graph augmentation + GNN forward + FAISS)
# ---------------------------------------------------------------------------

@torch.no_grad()
def cold_start_recommend(
    model: torch.nn.Module,
    index,
    new_features: torch.Tensor,
    g: dgl.DGLGraph,
    existing_features: torch.Tensor,
    k_nn: int = 5,
    k: int = 10,
    device: str = "cuda",
) -> Tuple[np.ndarray, np.ndarray]:
    """Cold-start recommendation using paper's approach (Virinchi et al., 2022):
    1. k-NN feature lookup to find similar existing products.
    2. Add bidirectional co-view edges between cold node and neighbours.
    3. Run GNN forward pass on augmented graph for cold node's source embedding.
    4. Search FAISS index with cold node's source embedding for top-k recommendations."""
    model = model.to(device).eval()

    if new_features.dim() == 1:
        new_features = new_features.unsqueeze(0)

    # Step 1: k-NN feature lookup
    nn_model = NearestNeighbors(n_neighbors=k_nn, metric="cosine")
    nn_model.fit(existing_features.cpu().numpy())
    _distances, knn_indices = nn_model.kneighbors(new_features.cpu().numpy().reshape(1, -1))
    similar_ids = knn_indices[0]

    # Step 2: Augment graph with cold-start edges
    cs_id = g.num_nodes()
    g_aug = dgl.add_nodes(g, 1)
    g_aug.ndata["feat"] = torch.cat([g.ndata["feat"], new_features.unsqueeze(0).cpu()])
    for sid in similar_ids:
        g_aug.add_edges(cs_id, int(sid), {"type": torch.tensor([1])})
        g_aug.add_edges(int(sid), cs_id, {"type": torch.tensor([1])})

    # Step 3: GNN forward pass on augmented graph
    num_layers = len(model.layers) if hasattr(model, "layers") else 3
    sampler = MultiLayerNeighborSampler([20, 10, 10][:num_layers])
    loader = DGLDataLoader(
        g_aug, torch.arange(g_aug.num_nodes()), sampler,
        batch_size=4096, shuffle=False, device=device,
    )

    all_src: List[torch.Tensor] = []
    for _input_nodes, _output_nodes, blocks in loader:
        blocks = [b.to(device) for b in blocks]
        h = blocks[0].srcdata["feat"].to(device)
        out = model(blocks, h)
        if isinstance(out, (tuple, list)) and len(out) == 2:
            all_src.append(out[0].cpu())
        else:
            all_src.append(out.cpu())

    src_emb = torch.cat(all_src, dim=0)

    # Step 4: Search with cold node's source embedding
    query = src_emb[cs_id].cpu().numpy().reshape(1, -1).astype(np.float32)
    query = sk_normalize(query, norm="l2")
    distances, indices = index.search(query, k)

    return indices[0], distances[0]


print("Evaluation functions defined")


In [ ]:
# Cell 8: Configuration
cfg = DAEMONConfig(
    in_feats=384, hidden_dim=128, out_dim=64, num_layers=3, dropout=0.1,
    epochs=30, batch_size=1024, num_neighbors=(20, 10, 10),
    lr=1e-4, weight_decay=1e-5, use_amp=True, patience=5, num_neg=5,
    output_dir="/kaggle/working/", checkpoint_path="/kaggle/working/daemon_best.pt",
)

# Local fallback
if not os.path.isdir("/kaggle/working"):
    cfg.output_dir = "./output/"
    os.makedirs(cfg.output_dir, exist_ok=True)
    cfg.checkpoint_path = "./output/daemon_best.pt"

print("Configuration:")
for f in sorted(cfg.__dataclass_fields__):
    print(f"  {f}: {getattr(cfg, f)}")


In [ ]:
# Cell 9: Data loading (synthetic, 50K nodes)
RUN_SYNTHETIC = True  # Set False for real Kaggle data

if RUN_SYNTHETIC:
    print("Generating synthetic product graph (50K nodes)...")
    cp_edges, cv_edges, features, categories = generate_synthetic_graph(
        num_products=50000, feature_dim=cfg.in_feats,
        avg_cp_degree=5, avg_cv_degree=8, asymmetry_ratio=0.75
    )
    np.random.seed(42)
    print(f"CP edges: {cp_edges.shape[1]:,}, CV edges: {cv_edges.shape[1]:,}")
else:
    # Real Kaggle data path — update with your dataset
    import kagglehub
    path = kagglehub.dataset_download("your-dataset/daemon-products")
    features = np.load(f"{path}/features.npy").astype(np.float32)
    cp_edges = np.load(f"{path}/cp_edges.npy")
    cv_edges = np.load(f"{path}/cv_edges.npy")
    categories = np.zeros(features.shape[0], dtype=np.int64)

# Build DGL graph
product_df = pd.DataFrame({
    'title': [f'Product_{i}' for i in range(features.shape[0])],
})

g = build_product_graph(product_df, cp_edges, cv_edges, features=features, feature_dim=cfg.in_feats)
cfg.num_nodes = g.num_nodes()
cfg.num_edges = g.num_edges()
print_graph_stats(g)
validate_graph(g)

# Save graph
try:
    dgl.save_graphs(f"{cfg.output_dir}/product_graph.bin", [g])
    print(f"Graph saved to {cfg.output_dir}")
except Exception as e:
    print(f"Could not save graph: {e}")


In [ ]:
# Cell 10: Graph splitting + DataLoaders
split = split_edges_by_type(g, train_ratio=0.75, val_ratio=0.05)
print(f"Train CP: {len(split['train_cp']):,}, CV: {len(split['train_cv']):,}")
print(f"Val CP:   {len(split['val_cp']):,}, CV: {len(split['val_cv']):,}")
print(f"Test CP:  {len(split['test_cp']):,}, CV: {len(split['test_cv']):,}")

# Create training subgraph
train_eids = torch.cat([split['train_cp'], split['train_cv']])
train_g = dgl.edge_subgraph(g, train_eids, relabel_nodes=False)

# Find one-way edges
one_way_u, one_way_v = find_one_way_edges(train_g)
print(f"One-way CP edges: {len(one_way_u):,}")

# Create DataLoaders
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sampler = MultiLayerNeighborSampler(list(cfg.num_neighbors))
train_seeds = torch.arange(cfg.num_nodes)
train_loader = DGLDataLoader(
    train_g, train_seeds, sampler,
    batch_size=cfg.batch_size, shuffle=True, drop_last=False,
    num_workers=0, device=device
)
print(f"Training batches: {len(train_loader)}")

# Test single batch
test_inputs, test_outputs, test_blocks = next(iter(train_loader))
test_blocks = [b.to(device) for b in test_blocks]
print(f"Subgraph: {test_blocks[0].num_src_nodes()} src -> {test_blocks[-1].num_dst_nodes()} dst")
cp_n = (test_blocks[-1].edata['type'] == 0).sum().item()
cv_n = (test_blocks[-1].edata['type'] == 1).sum().item()
print(f"Edges: {cp_n} CP + {cv_n} CV")
memory_summary()


In [ ]:
# Cell 11: Model instantiation + smoke test
model = DAEMONModel(cfg)
criterion = AsymmetricLoss(cfg)
model = model.to(device)

n_params = count_parameters(model)
print(f"DAEMON parameters: {n_params:,}")
print(f"Model size: {sum(p.numel() * 4 for p in model.parameters() if p.requires_grad) / 1e6:.1f} MB")
memory_summary()

# Smoke test forward pass
model.train()
test_blocks_gpu = [b.to(device) for b in test_blocks]
h = test_blocks_gpu[0].srcdata['feat']

with autocast(enabled=cfg.use_amp):
    src_emb, tgt_emb = model(test_blocks_gpu, h)
    block = test_blocks_gpu[-1]
    cp_mask = block.edata['type'] == 0
    cv_mask = block.edata['type'] == 1
    cp_eids = torch.where(cp_mask)[0]
    cv_eids = torch.where(cv_mask)[0]

    if len(cp_eids) > 0 and len(cv_eids) > 0:
        cp_src, cp_dst = block.find_edges(cp_eids[:100])
        cv_src, cv_dst = block.find_edges(cv_eids[:100])
        loss, components = criterion(
            src_emb, tgt_emb,
            cp_u=cp_dst[:50], cp_v=cp_dst[:50],
            ow_u=cp_dst[:20], ow_v=cp_dst[:20],
            cv_u=cv_dst[:50], cv_v=cv_dst[:50],
            num_nodes=tgt_emb.size(0)
        )
        print(f"Smoke test OK - Loss: {loss.item():.4f}")
        print(f"  CP: {components['cp'].item():.4f}")
        print(f"  Asym: {components['ow'].item():.4f}")
        print(f"  CV: {components['cv'].item():.4f}")
    else:
        print("Smoke test skipped - insufficient edge types in batch")

del test_blocks_gpu, h; gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Cell 12: Training setup + checkpoint resume check
setup = setup_training(cfg)
model = setup['model']
optimizer = setup['optimizer']
criterion_wrapped = setup['criterion']
scheduler = setup['scheduler']
scaler = setup['scaler']

# Checkpoint resume
resume_path = cfg.checkpoint_path
start_epoch = 0
if os.path.isfile(resume_path):
    start_epoch, _ = load_checkpoint(
        resume_path, model=model, optimizer=optimizer,
        scaler=scaler, scheduler=scheduler
    )
    print(f"Resumed from checkpoint at epoch {start_epoch}")
else:
    print("No checkpoint found - starting fresh training")


In [ ]:
# Cell 13: Full training
history = train_model(
    model=model, train_loader=train_loader, g=g,
    val_cp_eids=split['val_cp'],
    criterion=criterion_wrapped, optimizer=optimizer,
    scheduler=scheduler, scaler=scaler, cfg=cfg,
    resume_epoch=start_epoch
)
print("Training complete")


In [ ]:
# Cell 14: Training history plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(history['train_loss'], label='Train Loss', color='#2196F3')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
if 'val_auc' in history and history['val_auc']:
    ax.plot(history['val_auc'], label='Val AUC', color='#4CAF50', marker='o')
if 'val_hr10' in history:
    ax.plot(history['val_hr10'], label='Val HR@10', color='#FF9800', marker='s')
ax.set_xlabel('Epoch'); ax.set_ylabel('Metric')
ax.set_title('Validation Metrics'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.output_dir}/training_history.png", dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 15: Load best model
best_path = cfg.checkpoint_path
if os.path.isfile(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print("Loaded best model")
else:
    print("Using current model state")


In [ ]:
# Cell 16: Generate embeddings for all products
print("Generating embeddings for all products...")
src_emb, tgt_emb = generate_all_embeddings(model, g, batch_size=4096, device=device)
print(f"Source: {src_emb.shape}, Target: {tgt_emb.shape}")
memory_summary()


In [ ]:
# Cell 17: Full evaluation
results = {}
cp_test_src, cp_test_dst = g.find_edges(split['test_cp'])

# HitRate@k and MRR@k
for k in cfg.hitrate_k:
    results[f'HR@{k}'] = compute_hit_rate_at_k(src_emb, tgt_emb, cp_test_src, cp_test_dst, k=k)
    results[f'MRR@{k}'] = compute_mrr_at_k(src_emb, tgt_emb, cp_test_src, cp_test_dst, k=k)

# Link prediction AUC
n_test = len(cp_test_src)
neg_src = torch.randint(0, cfg.num_nodes, (n_test,))
neg_dst = torch.randint(0, cfg.num_nodes, (n_test,))
results['AUC'] = compute_link_prediction_auc(
    src_emb, tgt_emb,
    torch.stack([cp_test_src, cp_test_dst], dim=1),
    torch.stack([neg_src, neg_dst], dim=1)
)

# Direction AUC
n_ow = min(10000, len(one_way_u))
results['Direction_AUC'] = compute_direction_prediction_auc(
    src_emb, tgt_emb,
    torch.stack([one_way_u[:n_ow], one_way_v[:n_ow]], dim=1)
)

# Print results
print("=" * 60)
print("DAEMON Evaluation Results")
print("=" * 60)
for k in cfg.hitrate_k:
    print(f"  HitRate@{k:2d}: {results[f'HR@{k}']:.4f}")
    print(f"  MRR@{k:2d}:     {results[f'MRR@{k}']:.4f}")
print(f"  AUC:            {results['AUC']:.4f}")
print(f"  Direction AUC:  {results['Direction_AUC']:.4f}")
print("=" * 60)

# Save metrics
with open(f"{cfg.output_dir}/metrics.json", 'w') as f:
    json.dump({k: float(v) if isinstance(v, (torch.Tensor, np.floating, np.ndarray)) else v
               for k, v in results.items()}, f, indent=2)


In [ ]:
# Cell 18: Build FAISS index + latency test
try:
    faiss_index = build_faiss_index(tgt_emb, use_gpu=torch.cuda.is_available())
    print(f"FAISS index: {faiss_index.ntotal:,} vectors x {tgt_emb.shape[1]} dims")
    use_faiss = True
except Exception as e:
    print(f"FAISS failed ({e}), using brute-force search")
    use_faiss = False

if use_faiss:
    query = src_emb[:100].cpu().numpy().astype(np.float32)
    _ = faiss_index.search(query, 10)
    t0 = time.time()
    for _ in range(10):
        faiss_index.search(query, 10)
    latency = (time.time() - t0) / 10 / 100 * 1000
    print(f"FAISS latency: {latency:.2f} ms/query")
    if latency < 100:
        print("Latency OK (< 100ms)")

# Save FAISS index
if use_faiss:
    try:
        cpu_index = faiss.index_gpu_to_cpu(faiss_index) if hasattr(faiss, 'index_gpu_to_cpu') else faiss_index
        faiss.write_index(cpu_index, f"{cfg.output_dir}/faiss_index.bin")
        print("FAISS index saved")
    except Exception as e:
        print(f"Could not save FAISS index: {e}")


In [ ]:
# Cell 19: Recommendation demo
sample_queries = list(range(min(10, cfg.num_nodes)))

for q_id in sample_queries[:5]:
    print(f"Query: Product_{q_id}")
    if use_faiss:
        recs = recommend_related(src_emb, faiss_index, q_id, k=5)
        for rank, (rec_id, score) in enumerate(zip(recs[0], recs[1]), 1):
            print(f"  {rank}. Product_{rec_id} [{score:.3f}]")
    else:
        q_vec = src_emb[q_id]
        scores = q_vec @ tgt_emb.T
        scores[q_id] = -float('inf')
        top_k = torch.topk(scores, 5)
        for rank, (rec_id, score) in enumerate(zip(top_k.indices, top_k.values), 1):
            print(f"  {rank}. Product_{rec_id.item()} [{score.item():.3f}]")
    print()


In [ ]:
# Cell 20: Cold-start demo
print("Simulating cold-start product...")
new_feat = torch.randn(cfg.in_feats)
existing_feats = g.ndata['feat'].cpu()

try:
    if use_faiss:
        cold_recs = cold_start_recommend(
            model=model, index=faiss_index,
            new_features=new_feat, g=g, existing_features=existing_feats,
            k_nn=5, k=10, device=device
        )
    else:
        # Fallback without FAISS: use feature similarity + brute-force
        feat_sim = F.cosine_similarity(new_feat.unsqueeze(0), existing_feats.float())
        top10 = torch.topk(feat_sim, 10)
        cold_recs = (top10.indices.numpy(), top10.values.numpy())

    print("Cold-Start Recommendations:")
    for rank, rec_id in enumerate(cold_recs[0][:5], 1):
        print(f"  {rank}. Product_{rec_id}")
except Exception as e:
    print(f"Cold-start failed: {e}")
    print("Using feature similarity fallback...")
    feat_sim = F.cosine_similarity(new_feat.unsqueeze(0), existing_feats.float())
    top5 = torch.topk(feat_sim, 5)
    print("Top-5 by feature similarity:")
    for rank, (idx, sim) in enumerate(zip(top5.indices, top5.values), 1):
        print(f"  {rank}. Product_{idx.item()} [{sim.item():.3f}]")


In [ ]:
# Cell 21: Ablation studies (OPTIONAL — set RUN_ABLATION=True to run)
RUN_ABLATION = False

if RUN_ABLATION:
    ablation_results = {}
    for name in ['Full DAEMON', 'w/o Asymmetry']:
        print(f"Training {name}...")
        ab_model = DAEMONModel(cfg).to(device)
        ab_optimizer = torch.optim.Adam(ab_model.parameters(), lr=cfg.lr)
        for ep in range(1, 6):
            ab_model.train()
            for it, (inp, out, blocks) in enumerate(train_loader):
                blocks = [b.to(device) for b in blocks]
                h = blocks[0].srcdata['feat']
                with autocast(enabled=cfg.use_amp):
                    s, t = ab_model(blocks, h)
                    cp_mask = blocks[-1].edata['type'] == 0
                    if cp_mask.sum() == 0: continue
                    cp_eids = torch.where(cp_mask)[0]
                    cp_s, cp_d = blocks[-1].find_edges(cp_eids[:50])
                    loss, _ = criterion(s, t, cp_s, cp_d, cp_s[:10], cp_d[:10],
                                       cp_s[:10], cp_d[:10], t.size(0))
                loss.backward()
                ab_optimizer.step()
                ab_optimizer.zero_grad()
                if it >= 5: break
        ab_src, ab_tgt = generate_all_embeddings(ab_model, g, batch_size=4096, device=device)
        hr10 = compute_hit_rate_at_k(ab_src, ab_tgt, cp_test_src, cp_test_dst, k=10)
        ablation_results[name] = {'HR@10': hr10}
        del ab_model; gc.collect(); torch.cuda.empty_cache()

    ab_df = pd.DataFrame(ablation_results).T
    print(ab_df)
else:
    print("Ablation studies skipped. Set RUN_ABLATION=True to run.")


In [ ]:
# Cell 22: Export artifacts
# Save model checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg,
    'results': results,
}, f"{cfg.output_dir}/daemon_model.pt")
print(f"Model saved to {cfg.output_dir}/daemon_model.pt")

# Save embeddings
np.save(f"{cfg.output_dir}/src_embeddings.npy", src_emb.cpu().numpy().astype(np.float16))
np.save(f"{cfg.output_dir}/tgt_embeddings.npy", tgt_emb.cpu().numpy().astype(np.float16))
print(f"Embeddings saved ({src_emb.shape})")

# Final summary
print("=" * 60)
print("  DAEMON - Final Results Summary")
print("=" * 60)
for k, v in results.items():
    if isinstance(v, (int, float, np.floating, np.ndarray)):
        print(f"  {k}: {float(v):.4f}")
    else:
        print(f"  {k}: {v}")

print(f"Output directory: {cfg.output_dir}")
for fname in ['daemon_model.pt', 'src_embeddings.npy', 'tgt_embeddings.npy',
              'metrics.json', 'faiss_index.bin', 'training_history.png']:
    path = os.path.join(cfg.output_dir, fname)
    if os.path.isfile(path):
        print(f"  {fname} ({os.path.getsize(path)/1e6:.1f} MB)")

print("To download: Kaggle UI > Notebook > Output > Download All")
memory_summary()
print("Notebook execution complete!")
